Q1. Data Understanding

Identify all data quality issues present in the dataset that can cause problems during data loading

A) Common Data Quality Issues Before Loading
1. Missing Data

    - Null or blank values in critical fields (e.g., Customer ID, Transaction Date).

    - Entire columns with sparse data, making imputation unreliable.

    - Causes: system errors, optional fields, or incomplete user input.

2. Duplicate Records

    - Multiple entries for the same entity (e.g., same customer appearing twice).

    - Integration overlaps when merging datasets from different sources.

    - Leads to inflated counts and distorted KPIs.

3. Inconsistent Formats

    - Date formats (DD/MM/YYYY vs MM-DD-YYYY).

    - Country codes (IN, IND, India).

    - Numeric fields stored as text ("1000" vs 1000).

    - Breaks schema validation during loading.

4. Invalid Values

    - Out-of-range entries (negative ages, impossible dates).

    - Violations of business rules (discount > 100%, salary = 0).

    - Causes downstream errors in aggregations.

5. Referential Integrity Issues

    - Foreign keys not matching primary keys (e.g., Order referencing a non-existent Customer).

    - Broken relationships between tables.

6. Inconsistent Units or Scales

    - Currency mismatch (USD vs INR).

    - Measurement units (kg vs lbs).

    - Causes incorrect aggregations and misleading analytics.

7. Stale or Outdated Data

    - Records not updated in sync with source systems.

    - Leads to dashboards reflecting old realities.

8. Encoding & Special Characters

    - Non-UTF8 characters breaking ingestion.

    - Hidden spaces, tabs, or symbols in text fields.

9. Schema Drift

    - Source system changes (new columns, renamed fields).

    - Causes ETL jobs to fail during loading.




---


Q2. Primary Key Validation

Assume Order_ID is the Primary Key.

  a) Is the dataset violating the Primary Key rule?

  b) Which record(s) cause this violation?

A) Primary Key Validation

- A Primary Key must be unique and non-null across all rows. In your dataset, Order_ID is repeated, which violates uniqueness.


- The duplicate records are:

| Order_ID | Customer_ID | Sales_Amount | Order_Date |
| --- | --- | --- | --- |
| O101 | C001 | 4500 | 12-01-2024 |
| O101 | C001 | 4500 | 12-01-2024 |

Both rows have the same Order_ID = O101.

This duplication breaks the Primary Key constraint.



---


Q3.  Missing Value Analysis

Which column(s) contain missing values?

a) List the affected records.

b) Explain why loading these records without handling missing values is risky.

A) a) Affected Records
The record with the missing value is:

| Order_ID | Customer_ID | Sales_Amount | Order_Date |
| --- | --- | --- | --- |
| O102 | C002 | **Null** | 15-01-2024 |

b) Why Loading Without Handling is Risky

1. Breaks Aggregations

  - Summations, averages, or totals on Sales_Amount will fail or produce incorrect results.

- Example: Total sales will be understated if nulls are treated as zero.

2. Schema Validation Errors

- If the column is defined as NOT NULL in the database schema, the load will fail.

3. Distorted Analytics

- Missing values can bias KPIs (e.g., average sales per customer).

- Business decisions may be based on incomplete data.

4. Downstream ETL Failures

- Joins, transformations, or calculations relying on Sales_Amount may throw errors or propagate nulls.



---


Q4. Data Type Validation

Identify records where Sales_Amount violates expected data type rules.

a) Which record(s) will fail numeric validation?

b) What would happen if this dataset is loaded into a SQL table with Sales_Amaaount as DECIMAL?

A) Records Violating Numeric Validation
The column Sales_Amount is expected to be numeric (e.g., DECIMAL).

a) Records that fail numeric validation:

| Order_ID | Customer_ID | Sales_Amount | Order_Date |
| --- | --- | --- | --- |
| O104 | C004 | **Three Thousand** | 20-01-2024 |

"Three Thousand" is stored as text instead of a number.

This violates the numeric data type rule.

b) What happens if loaded into SQL with Sales_Amount as DECIMAL?

1. Load Failure / Error

- SQL will reject non-numeric strings when inserting into a DECIMAL column.

- Example error: “Incorrect DECIMAL value: 'Three Thousand'”.

2. ETL Job Crash

- The pipeline may halt if strict type enforcement is enabled.

- Downstream transformations depending on numeric operations will fail.

3. Data Loss (if forced conversion)

- If implicit conversion is attempted, the value may be stored as 0 or cause truncation.

- This leads to inaccurate reporting.



---

Q5.  Date Format Consistency

The Order_date column has multiple formats.

a) List all date formats present in the dataset

b) Why is this a problem during data loading?

A)
1. Date Formats Present
The Order_Date column contains multiple formats:

- 12-01-2024 → DD-MM-YYYY (or ambiguous MM-DD-YYYY).

- 15-01-2024 → same DD-MM-YYYY style.

- 2024/01/18 → YYYY/MM/DD.

- 20-01-2024 → DD-MM-YYYY.

- 25-01-2024 → DD-MM-YYYY.

So, the dataset mixes dash-separated day-first dates with slash-separated year-first dates.

b) Why This Is a Problem During Data Loading

1. Parsing Errors

- ETL tools or SQL engines may fail to interpret inconsistent formats.

- Example: 12-01-2024 could be read as 12 Jan 2024 or Jan 12 2024 depending on locale.

2. Schema Validation Failures

- If the column is defined as DATE, non-standard formats (YYYY/MM/DD) may be rejected.

3. Inconsistent Sorting & Filtering

- Queries like ORDER BY Order_Date or WHERE Order_Date BETWEEN … will misbehave if formats aren’t standardized.

4. Ambiguity in Business Reporting

- Dash format (12-01-2024) is ambiguous across regions (India vs US).

- This can lead to incorrect reporting of timelines.



---

Q6. Load Readiness Decision

Based on the dataset condition:

a) Should this dataset be loaded directly into the database? (Yes/No)

b) Justify your answer with at least three reasons

A)
  a) Should this dataset be loaded directly into the database?

No.

b) Justification (at least three reasons)

1. Primary Key Violation

- Order_ID = O101 is duplicated, breaking the uniqueness rule of a primary key.

- This will cause constraint errors during loading.

2. Missing Values

- Sales_Amount for Order_ID = O102 is Null.

- If the column is defined as NOT NULL, the load will fail. Even if allowed, analytics will be distorted.

3. Data Type Inconsistency

- Sales_Amount = "Three Thousand" (text instead of numeric) for Order_ID = O104.

- This will fail insertion into a DECIMAL column.

4. Date Format Inconsistency

- Mixed formats (DD-MM-YYYY vs YYYY/MM/DD) in Order_Date.

- Causes parsing errors and schema validation failures.



---

Q7. Pre-Load Validation Checklist

List the exact pre-load validation checks you would perform on this dataset before loading.

A) Pre-Load Validation Checklist

1. Primary Key Validation

- Ensure Order_ID is unique and non-null.

- Detect and remove duplicates (e.g., O101 appears twice).

2. Missing Value Check

- Identify nulls in critical columns (Sales_Amount for O102).

- Decide handling strategy: imputation, default value, or exclusion.

3. Data Type Validation

- Verify Sales_Amount is numeric (DECIMAL).

- Flag invalid entries like "Three Thousand".

4. Date Format Standardization

- Convert all Order_Date values to a consistent format (e.g., ISO YYYY-MM-DD).

- Handle mixed formats (12-01-2024 vs 2024/01/18).

5. Range & Business Rule Validation

- Check Sales_Amount is positive and within expected limits.

- Ensure no unrealistic values (e.g., negative sales).

6. Referential Integrity Check

- Validate Customer_ID values against the master customer table.

- Ensure no orphan records.

7. Encoding & Special Character Check

- Confirm all text fields use valid encoding (UTF-8).

- Remove hidden spaces or symbols.

8. Schema Compliance

- Verify column names, data types, and constraints match the target database schema.

- Ensure no schema drift (extra/missing columns).



---

Q8. Cleaning Strategy

Describe the step-by-step cleaning actions required to make this dataset load-ready.

A)
1. Remove Duplicate Records
Ensure primary key uniqueness by eliminating repeated entries.

- Identify duplicates using GROUP BY Order_ID HAVING COUNT(*) > 1

- Keep the first occurrence or apply business rules to decide which record to retain

- Drop duplicate rows before loading

2. Handle Missing Values

Resolve nulls in critical fields to prevent schema errors.

- Detect nulls in Sales_Amount

- Apply imputation (e.g., median or average) or remove affected records

- Add a flag column (Sales_missing) if imputation is used

3. Fix Data Type Issues

- Convert non-numeric values in numeric fields.

- Locate invalid entries like Three Thousand

- Map text to numeric equivalents (e.g., replace with 3000)

- Use CAST or CONVERT to enforce DECIMAL type

4. Standardize Date Formats

- Ensure all dates follow a consistent format for reliable parsing.

- Convert mixed formats (DD-MM-YYYY, YYYY/MM/DD) into ISO YYYY-MM-DD

- Use SQL functions like STR_TO_DATE or ETL parsing rules

- Validate that all dates are valid calendar dates

5. Validate Referential Integrity

- Confirm foreign keys match master data to avoid orphan records.

- Cross-check Customer_ID values against the customer master table

- Remove or correct records with invalid references

- Enforce foreign key constraints in the schema

6. Final Schema Compliance Check

- Verify dataset matches target database schema before loading.

- Ensure column names and data types align with database definitions

- Confirm constraints (NOT NULL, PRIMARY KEY) are satisfied

- Run a test load into a staging table to validate readiness



---

Q9  Loading Strategy Selection

Assume this dataset represents daily sales data.

a) Should a Full Load or Incremental Load be used?

b) Justify your choice.

A) a) Should a Full Load or Incremental Load be used?
Incremental Load.

b) Justification

1. Nature of Daily Sales Data

- Sales transactions are generated continuously each day.

- Reloading the entire dataset (Full Load) would be redundant and inefficient.

2.  Performance & Efficiency

- Incremental Load processes only new or updated records since the last load.

- This reduces ETL runtime, minimizes resource usage, and avoids reprocessing historical data.

3. Data Integrity & Auditability

- Incremental loading preserves historical records untouched.

- Only new transactions (e.g., today’s sales) are appended, ensuring audit trails remain intact.

4. Reduced Risk of Errors

- Full Load risks overwriting or duplicating existing data if not carefully managed.

-  Incremental Load avoids duplication by checking for new Order_IDs or using Order_Date as a change detection key.



---

Q10. BI Impact Scenario

Assume this dataset was loaded without cleaning and connected to a BI dashboard.

a) What incorrect results might appear in Total Sales KPI?

b) Which records specifically would cause misleading insights?

c) Why would BI tools not detect these issues automatically?

A) a) Incorrect Results in Total Sales KPI

- Inflated totals due to duplicate records (O101 appears twice).

- Understated totals because Sales_Amount = Null for O102 is excluded from aggregation.

- Load failure or distortion if "Three Thousand" is treated as 0 or rejected, leading to missing sales.

- Date misalignment if inconsistent formats cause records to be excluded from time-based aggregations.

b) Records Causing Misleading Insights

1. Duplicate Record

- Order_ID = O101 (appears twice) → inflates sales by +4500.

2. Missing Value

- Order_ID = O102 (Sales_Amount = Null) → reduces total sales.

3. Invalid Data Type

- Order_ID = O104 (Sales_Amount = "Three Thousand") → fails numeric aggregation or misreported as 0.

4. Inconsistent Date Format

- Order_ID = O103 (Order_Date = 2024/01/18) → may be excluded from time-series charts.

c) Why BI Tools Won’t Detect These Issues Automatically

1. BI tools assume clean data

- They aggregate whatever is loaded without validating business rules.

2. No built-in semantic checks

- Tools don’t know "Three Thousand" is invalid unless preprocessing is applied.

3. Duplicates treated as valid rows

- BI dashboards will happily sum duplicate records, inflating KPIs.

4. Date parsing depends on source system

- Mixed formats may silently drop records or misplace them in wrong time buckets.


